# Finance Data Tutorial

Notebook ngắn này hướng dẫn 3 bước:
1. Bật cập nhật `financial` trong config updater.
2. Chạy updater để lưu parquet theo từng symbol.
3. Query dữ liệu tài chính bằng DuckDB để lọc cổ phiếu.

In [1]:
from vnstock_forecast.config import load_config
from vnstock_forecast.engine.schemas.config import to_app_config
from vnstock_forecast.engine.data.updater import update
from vnstock_forecast.engine.data.query import query_financial, query_sql

## 1) Bật financial updater bằng runtime override

Cấu trúc mới của updater:
- `data.updater.ohlcv.*`
- `data.updater.financial.*`

In [2]:
cfg = load_config(overrides=[
    "data.updater.financial.update=true",
    "data.updater.financial.symbols=[VHM,VNM,FPT]",
    "data.updater.ohlcv.update=false",
])
app_cfg = to_app_config(cfg)
print(app_cfg.data.updater)

UpdaterConfig(ohlcv=OhlcvUpdaterConfig(update=False, client=<DataClient.vietstock: 'vietstock'>, symbols=['ACB', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VPL', 'VRE'], resolutions=['D'], lookback_days=3600), financial=FinancialUpdaterConfig(update=True, client=<DataClient.vietcap: 'vietcap'>, symbols=['VHM', 'VNM', 'FPT']))


## 2) Chạy updater

Khi chạy thành công, dữ liệu được lưu theo cấu trúc:
- `data/finance/<SYMBOL>/cash_flow.parquet`
- `data/finance/<SYMBOL>/income_statement.parquet`
- `data/finance/<SYMBOL>/balance_sheet.parquet`
- `data/finance/<SYMBOL>/footnote.parquet`
- `data/finance/<SYMBOL>/statistics.parquet`

In [3]:
ok = update(app_cfg.data.updater)
print("Update status:", ok)

Update status: True


## 3) Query tài chính để lọc cổ phiếu

`query_financial` trả dữ liệu long-format: `symbol, statement, metric, description, period, value`.

In [4]:
df = query_financial(
    symbols=["VHM", "VNM", "FPT"],
    statements="statistics",
    metrics=["roe", "roa"],
    periods="Q4_2025",
    min_value=0.1,
)
df.head(20)

,symbol,statement,metric,description,period,value
0,FPT,statistics,roa,,Q4_2025,0.117051
1,FPT,statistics,roe,,Q4_2025,0.282717
2,VHM,statistics,roe,,Q4_2025,0.187345
3,VNM,statistics,roa,,Q4_2025,0.173682
4,VNM,statistics,roe,,Q4_2025,0.298906


In [5]:
# Ví dụ SQL screening: lấy mã có ROE Q4_2025 > 0.15
screen_df = query_sql(
    """
    SELECT symbol, value AS roe
    FROM finance_long
    WHERE statement = 'statistics'
      AND metric = 'roe'
      AND period = 'Q4_2025'
      AND value > 0.15
    ORDER BY roe DESC
    """
)
screen_df

,symbol,roe
0,SCS,0.506201
1,GEE,0.449878
2,BMP,0.440473
3,VCG,0.397973
4,ANV,0.316112
5,CTR,0.304960
6,VNM,0.298906
7,VIX,0.288538
8,SIP,0.284709
9,FPT,0.282717
